# CFRI doğrulayıcı çalışma: fold D (2 Ocak ile 30 Eylül 2026 arası)

**Çalıştırmadan önce:** `cfri_confirmatory/` klasörünü (protokol `CFRI_CONFIRMATORY_PROTOCOL_v1.md`, `code/`, bu notebook)
GitHub deposuna commit edin. Commit zamanı, protokolün 2026 verisi indirilmeden önce dondurulduğunun kanıtıdır. Bu
notebook kod dosyalarını protokolde yazan SHA-256 değerleriyle doğrular; tek bayt farkı olursa durur.

**Çalışma ortamı:** Runtime > Change runtime type > **GPU (tercihen A100)**, sonra **Runtime > Run all**.

Notebook sırasıyla şunları yapar:

1. Sabit sürümlü paketleri kurar.
2. Dondurulmuş kodu yazar ve SHA-256 ile doğrular.
3. Google Drive'ı bağlar. Her şey `MyDrive/CFRI_confirmatory` klasörüne yazılır.
4. **Veri:** 34 ETF için 1 Ocak 2015 ile 30 Eylül 2026 arası günlük OHLCV verisini Yahoo Finance'ten indirir ve
   protokol v1'in kalite kurallarını uygular. Bu adımda hiçbir yöntem çalışmaz. Ham veri yalnızca sizin Drive'ınızda
   kalır (`raw_do_not_publish` klasörü); bu klasörü paylaşmayın ve GitHub'a yüklemeyin.
5. **CPU yöntemleri:** CFRI, log-lineer, ITRI, lineer, chained ridge ve OHLC projeksiyonlu sürümleri, iki maske
   türünde. A100 çalışma ortamında yaklaşık 5 ile 15 dakika.
6. **SAITS:** önce tek koşulluk duman testi, sonra V2 ve V1, her biri iki maske türünde. İşlemci sayısı yeterliyse
   dört süreç aynı anda çalışır. A100'de yaklaşık 3 ile 5 saat. İlerleme her dakika yazdırılır.
7. **Analiz:** önceden belirlenmiş H1, H2, H3, H4 ve ikincil analizler. Sonuç ZIP'i otomatik iner
   (`CFRI_fold_D_results.zip`). ZIP ham fiyat içermez.

Oturum kesilirse aynı notebook'u yeniden **Run all** yapın: tamamlanmış parçalar SHA-256 ile doğrulanıp atlanır,
kalan yerden devam edilir. Veri bir kez indirildikten sonra tekrar indirilmez.

Bitince `CFRI_fold_D_results.zip` dosyasını Claude'a gönderin.

## 1. Paketler

In [ ]:
%pip -q install yfinance==1.7.0 exchange_calendars==4.13.2 numpy==2.1.3 pandas==2.2.3 scipy==1.18.1 scikit-learn==1.6.1 pypots==1.5 pygrinder==0.7

## 2. Dondurulmuş kod (SHA-256 doğrulamalı)

In [ ]:
# @title Frozen code (expand to read; verified by SHA-256 before writing)
FILES = {}
FILES['cfri_core.py'] = r'''"""Cross-Feature Residual Interpolation (CFRI) and its comparators: frozen core for the confirmatory study.

CFRI is the method explored under the working name "ITRI2" (notebook ITRI_2_Colab_Drive_Resume.ipynb,
code version "itri2-residual-colab-v1"). Its price and volume rules, candidate grids, pseudo-mask seeds and
per-condition validation selection are reproduced here unchanged; the functions marked [verbatim] are copied
from that notebook. This module adds: a whole-row mask type, candidate-level logging (needed for the
fixed-setting ablation), an OHLC consistency diagnostic, an OHLC projection variant, and fold D.

Masks, folds A-C, scaling and the primary endpoint are identical to the external validation study
(external_validation/code/masking.py and run_classical_benchmark.py of the public repository).
"""

from __future__ import annotations

import hashlib
import math
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

CODE_ID = "cfri-core-v2"
PSEUDO_SEED_TAG = "itri2-residual-colab-v1"   # keeps CFRI pseudo masks identical to the exploratory ITRI2 run
ORIGINAL_MANIFEST_SHA = "37ed5c7becd54665c108ecf296bca1a0dd94ffc0a20788a3b9dd42679eb70d64"
FEATURES = ("open", "high", "low", "close", "volume")
ASSETS = (
    ("EWC", "Canada", "Americas"), ("EWZ", "Brazil", "Americas"),
    ("ECH", "Chile", "Americas"), ("EWW", "Mexico", "Americas"),
    ("EWO", "Austria", "Europe"), ("EWK", "Belgium", "Europe"),
    ("EDEN", "Denmark", "Europe"), ("EFNL", "Finland", "Europe"),
    ("EWQ", "France", "Europe"), ("EWG", "Germany", "Europe"),
    ("EIRL", "Ireland", "Europe"), ("EWI", "Italy", "Europe"),
    ("EWN", "Netherlands", "Europe"), ("ENOR", "Norway", "Europe"),
    ("EPOL", "Poland", "Europe"), ("EWP", "Spain", "Europe"),
    ("EWD", "Sweden", "Europe"), ("EWL", "Switzerland", "Europe"),
    ("EWU", "United Kingdom", "Europe"), ("EWA", "Australia", "Asia-Pacific"),
    ("MCHI", "China", "Asia-Pacific"), ("EWH", "Hong Kong", "Asia-Pacific"),
    ("INDA", "India", "Asia-Pacific"), ("EIDO", "Indonesia", "Asia-Pacific"),
    ("EWJ", "Japan", "Asia-Pacific"), ("EWM", "Malaysia", "Asia-Pacific"),
    ("ENZL", "New Zealand", "Asia-Pacific"), ("EPHE", "Philippines", "Asia-Pacific"),
    ("EWS", "Singapore", "Asia-Pacific"), ("EWY", "South Korea", "Asia-Pacific"),
    ("EWT", "Taiwan", "Asia-Pacific"), ("THD", "Thailand", "Asia-Pacific"),
    ("EIS", "Israel", "Middle East and Africa"),
    ("EZA", "South Africa", "Middle East and Africa"),
)
FOLDS = {
    "A": {"train": (2015, 2018), "val": (2019, 2019), "test": (2020, 2021)},
    "B": {"train": (2015, 2020), "val": (2021, 2021), "test": (2022, 2023)},
    "C": {"train": (2015, 2022), "val": (2023, 2023), "test": (2024, 2025)},
    # Confirmatory fold: the test year is 2026, used only up to the protocol cut-off date.
    "D": {"train": (2015, 2024), "val": (2025, 2025), "test": (2026, 2026)},
}
FOLD_D_TEST_END = "2026-09-30"
BLOCKS = (1, 2, 3, 5, 10, 20)
RATES = (0.10, 0.20, 0.30)
SEEDS = (11, 23, 37)
EDGE_GUARD = 20
MASK_TYPES = ("feature", "row")
RIDGE_ALPHAS = (0.1, 1.0, 10.0)
PRICE_GAINS = (0.0, 0.25, 0.5, 0.75, 1.0)
VOLUME_SHRINK = (0.0, 0.25, 0.5, 0.75)
PRICE_CANDIDATES = tuple((a, g) for a in RIDGE_ALPHAS for g in PRICE_GAINS)
VOLUME_CANDIDATES = ("linear_raw", "itri_w1_p5") + tuple(f"log_local_{e:.2f}" for e in VOLUME_SHRINK)
# Fixed CFRI setting, selected before the confirmatory study as the candidate pair and volume rule with the
# lowest mean validation NRMSE pooled over all feature-wise conditions of folds A-C (validation years 2019,
# 2021 and 2023; no test data involved).
CFRI_FIXED = (10.0, 0.75, "log_local_0.50")
BASE_METHODS = ("linear_raw", "log_linear", "itri_w1_p5", "chained_ridge", "cfri_adaptive_no_cross",
                "cfri_adaptive", "cfri_fixed_no_cross", "cfri_fixed")
METHODS = BASE_METHODS + tuple(m + "_proj" for m in BASE_METHODS)
PRIMARY_METHOD = "cfri_fixed_proj"    # CFRI as evaluated in the confirmatory study


# ----------------------------------------------------------------------------- masks and seeds
def run_lengths(mask: np.ndarray) -> np.ndarray:  # [verbatim]
    lengths = []
    for feature in range(mask.shape[1]):
        edges = np.flatnonzero(np.diff(np.r_[False, mask[:, feature], False]))
        lengths.extend((edges[1::2] - edges[::2]).tolist())
    return np.asarray(lengths, dtype=int)


def strict_block_mask(shape, rate, block_length, seed, edge_guard=EDGE_GUARD):  # [verbatim]
    rows, features = shape
    if rows <= 2 * edge_guard + block_length or features < 1:
        raise ValueError("partition too short for this mask")
    if not 0 < rate < 1 or block_length < 1 or edge_guard < 0:
        raise ValueError("invalid mask setting")
    eligible_rows = rows - 2 * edge_guard
    target = max(1, round(rate * eligible_rows * features))
    if target < features * block_length:
        raise ValueError("target too small for one run per feature")
    rng = np.random.default_rng(seed)
    mask = np.zeros(shape, dtype=bool)
    for feature in rng.permutation(features):
        start = int(rng.integers(edge_guard, rows - edge_guard - block_length + 1))
        mask[start:start + block_length, feature] = True
    remaining = target - features * block_length
    attempts = 0
    while remaining:
        length = min(block_length, remaining)
        feature = int(rng.integers(features))
        start = int(rng.integers(edge_guard, rows - edge_guard - length + 1))
        neighborhood = mask[max(edge_guard, start - 1):min(rows - edge_guard, start + length + 1), feature]
        if not neighborhood.any():
            mask[start:start + length, feature] = True
            remaining -= length
        attempts += 1
        if attempts >= 1000 * target:
            raise RuntimeError("could not place nonadjacent blocks")
    lengths = run_lengths(mask)
    if (int(mask.sum()) != target or lengths.max() > block_length or
            not np.all(mask.sum(axis=0) >= block_length) or
            mask[:edge_guard].any() or mask[rows - edge_guard:].any()):
        raise AssertionError("mask invariant failed")
    return mask


def stable_seed(ticker, fold, partition, block, rate, seed):  # [verbatim] (original study derivation)
    key = f"{ORIGINAL_MANIFEST_SHA}|{ticker}|{fold}|{partition}|{block}|{rate:.2f}|{seed}"
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def auxiliary_seed(ticker, fold, block, rate, seed, repetition):  # [verbatim] with the original tag
    key = f"{PSEUDO_SEED_TAG}|{ticker}|{fold}|pseudo|{block}|{rate:.2f}|{seed}|{repetition}"
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def condition_mask(shape, ticker, fold, partition, block, rate, seed, mask_type):
    """feature: each feature hidden independently (original generator and seeds).
    row: whole trading days hidden for all five features; the rows follow the same strict generator applied
    to one column, with seeds derived from the partition label suffixed by ':row'."""
    if mask_type == "feature":
        return strict_block_mask(shape, rate, block, stable_seed(ticker, fold, partition, block, rate, seed))
    if mask_type == "row":
        rows = strict_block_mask((shape[0], 1), rate, block,
                                 stable_seed(ticker, fold, partition + ":row", block, rate, seed))
        return np.repeat(rows, shape[1], axis=1)
    raise ValueError(mask_type)


def feasible(rows_by_part, block, rate, mask_type, edge_guard=EDGE_GUARD):
    """True when the strict generator can place the requested runs in every partition."""
    nfeat = len(FEATURES) if mask_type == "feature" else 1
    for rows in rows_by_part.values():
        if rows <= 2 * edge_guard + block:
            return False
        if max(1, round(rate * (rows - 2 * edge_guard) * nfeat)) < nfeat * block:
            return False
    return True


# ----------------------------------------------------------------------------- data helpers
def split_frame(frame: pd.DataFrame, fold: str) -> dict:
    parts = {}
    for name, (start, end) in FOLDS[fold].items():
        part = frame.loc[(frame.index.year >= start) & (frame.index.year <= end), list(FEATURES)]
        if fold == "D" and name == "test":
            part = part.loc[part.index <= pd.Timestamp(FOLD_D_TEST_END)]
        if len(part) < 100:
            raise ValueError(f"empty/short {fold}/{name}")
        parts[name] = part.to_numpy(float, copy=True)
    return parts


def observed_scaler(train, hidden):  # [verbatim]
    means = np.array([train[~hidden[:, d], d].mean() for d in range(train.shape[1])])
    scales = np.array([train[~hidden[:, d], d].std(ddof=0) for d in range(train.shape[1])])
    if not np.isfinite(means).all() or not np.isfinite(scales).all() or (scales <= 1e-12).any():
        raise ValueError("invalid training-only scaler")
    return means, scales


def corrupt(array, hidden):  # [verbatim]
    out = array.copy()
    out[hidden] = np.nan
    return out


def linear_fill(array):  # [verbatim]
    filled = pd.DataFrame(array).interpolate(method="linear", limit_direction="both")
    out = filled.ffill().bfill().to_numpy(float, copy=True)
    if not np.isfinite(out).all():
        raise ValueError("interpolation left nonfinite values")
    return out


class OriginalITRI:  # [verbatim]
    def fit(self, train_corrupt):
        mask = np.isnan(train_corrupt)
        filled = linear_fill(train_corrupt)
        self.models = []
        for d in range(filled.shape[1]):
            col = filled[:, d]
            z = np.column_stack((np.roll(col, 1), np.roll(col, -1)))
            z[[0, -1], :] = np.nan
            good = (~mask[:, d]) & np.isfinite(z).all(axis=1)
            if good.sum() < 4:
                self.models.append(None)
            else:
                design = np.column_stack((np.ones(good.sum()), z[good]))
                self.models.append(np.linalg.lstsq(design, col[good], rcond=None)[0])
        return self

    def transform(self, target_corrupt):
        mask = np.isnan(target_corrupt)
        out = linear_fill(target_corrupt)
        for _ in range(5):
            for d, beta in enumerate(self.models):
                if beta is None:
                    continue
                col = out[:, d]
                z = np.column_stack((np.roll(col, 1), np.roll(col, -1)))
                z[[0, -1], :] = np.nan
                rows = mask[:, d] & np.isfinite(z).all(axis=1)
                out[rows, d] = beta[0] + z[rows] @ beta[1:]
            out[~mask] = target_corrupt[~mask]
        return out


# ----------------------------------------------------------------------------- CFRI [verbatim functions]
def mask_runs(col_mask):
    edges = np.flatnonzero(np.diff(np.r_[False, col_mask, False]))
    return zip(edges[::2], edges[1::2])  # [start, stop)


def price_design(corrupt_log, target_feature, selected=None, truth_log=None):
    """Predictors use only visible cells; hidden target truth is used only as a label."""
    missing = np.isnan(corrupt_log)
    if selected is None:
        selected = missing[:, target_feature]
    xs, ys, rows, bases, supports = [], [], [], [], []
    n = len(corrupt_log)
    for start, stop in mask_runs(missing[:, target_feature]):
        a, b = start - 1, stop
        if a < 0 or b >= n:
            continue
        target_a, target_b = corrupt_log[a, target_feature], corrupt_log[b, target_feature]
        if not np.isfinite(target_a) or not np.isfinite(target_b):
            continue
        length = stop - start
        for t in range(start, stop):
            if not selected[t]:
                continue
            pos = (t - a) / (b - a)
            base = target_a + pos * (target_b - target_a)
            delta, availability = [], []
            for j in range(4):
                if j == target_feature:
                    continue
                visible = np.isfinite(corrupt_log[[a, t, b], j]).all()
                if visible:
                    aux_base = corrupt_log[a, j] + pos * (corrupt_log[b, j] - corrupt_log[a, j])
                    delta.append(float(corrupt_log[t, j] - aux_base))
                    availability.append(1.0)
                else:
                    delta.append(0.0)
                    availability.append(0.0)
            features = delta + availability + [math.log1p(length), pos, (target_b - target_a) / (b - a)]
            xs.append(features)
            rows.append(t)
            bases.append(base)
            supports.append(sum(availability))
            if truth_log is not None:
                ys.append(float(truth_log[t, target_feature] - base))
    x = np.asarray(xs, dtype=float).reshape((-1, 9))
    y = np.asarray(ys, dtype=float) if truth_log is not None else None
    return x, y, np.asarray(rows, dtype=int), np.asarray(bases), np.asarray(supports)


def prepare_price_training(train_raw, train_mask, ticker, fold, block, rate, seed):
    truth_log = np.log(train_raw[:, :4])
    datasets = [[] for _ in range(4)]
    labels = [[] for _ in range(4)]
    for repetition in range(2):
        pseudo = strict_block_mask(train_raw.shape, rate, block,
                                   auxiliary_seed(ticker, fold, block, rate, seed, repetition))
        effective = train_mask | pseudo
        visible_log = corrupt(truth_log, effective[:, :4])
        for d in range(4):
            selected = pseudo[:, d] & ~train_mask[:, d]
            x, y, _, _, _ = price_design(visible_log, d, selected, truth_log)
            if len(x):
                datasets[d].append(x)
                labels[d].append(y)
    return [(np.vstack(datasets[d]), np.concatenate(labels[d]))
            if datasets[d] else (np.empty((0, 9)), np.empty(0)) for d in range(4)]


def prepare_price_eval(raw, hidden):
    visible_log = np.log(corrupt(raw[:, :4], hidden[:, :4]))
    base_log = linear_fill(visible_log)
    designs = [price_design(visible_log, d) for d in range(4)]
    return base_log, designs


def fit_price_models(training, alpha):
    models = []
    for x, y in training:
        if len(y) < 30 or np.std(y) < 1e-10:
            models.append(None)
        else:
            scaler = StandardScaler().fit(x)
            model = Ridge(alpha=alpha).fit(scaler.transform(x), y)
            models.append((scaler, model))
    return models


def price_candidate(base_log, designs, models, gain):
    out = np.exp(base_log.copy())
    if gain <= 0:
        return out
    for d, pack in enumerate(models):
        if pack is None:
            continue
        x, _, rows, bases, supports = designs[d]
        if not len(rows):
            continue
        scaler, model = pack
        residual = np.clip(model.predict(scaler.transform(x)), -0.25, 0.25)
        residual[supports == 0] = 0.0
        out[rows, d] = np.exp(bases + gain * residual)
    return out


def volume_log_candidate(raw, hidden, eta, local_radius=20):
    visible = np.log1p(corrupt(raw[:, 4], hidden[:, 4]))
    base = linear_fill(visible[:, None])[:, 0]
    out = base.copy()
    for start, stop in mask_runs(hidden[:, 4]):
        neighborhood = visible[max(0, start - local_radius):min(len(visible), stop + local_radius)]
        nearby = neighborhood[np.isfinite(neighborhood)]
        if len(nearby):
            out[start:stop] = (1 - eta) * base[start:stop] + eta * np.median(nearby)
    result = np.expm1(out)
    result[~hidden[:, 4]] = raw[~hidden[:, 4], 4]
    return np.maximum(result, 0.0)


# ----------------------------------------------------------------------------- scoring, OHLC checks
def feature_rmse(truth, estimate, mask, scales, d):
    selected = mask[:, d]
    if selected.sum() == 0:
        raise ValueError("no masked evaluation cells")
    return float(np.sqrt(np.mean((estimate[selected, d] - truth[selected, d]) ** 2)) / scales[d])


def score_method(truth, estimate, hidden, scales):
    if estimate.shape != truth.shape or not np.isfinite(estimate).all():
        raise ValueError("invalid estimate")
    if not np.allclose(estimate[~hidden], truth[~hidden], rtol=1e-9, atol=1e-9):
        raise ValueError("method changed an observed cell")
    out = {}
    for d, feature in enumerate(FEATURES):
        selected = hidden[:, d]
        err = estimate[selected, d] - truth[selected, d]
        out[feature] = (float(np.sqrt(np.mean(err ** 2)) / scales[d]), float(np.mean(np.abs(err)) / scales[d]),
                        int(selected.sum()))
    return out


def ohlc_violations(estimate, hidden, tol=1e-9):
    """Share of rows with at least one imputed price cell that break high >= max(open, close), low <=
    min(open, close) or high >= low (relative tolerance)."""
    rows = hidden[:, :4].any(axis=1)
    if not rows.any():
        return 0.0, 0
    o, h, l, c = (estimate[rows, k] for k in range(4))
    bad = ((h < np.maximum(o, c) * (1 - tol)) | (l > np.minimum(o, c) * (1 + tol)) | (h < l * (1 - tol)))
    return float(bad.mean()), int(rows.sum())


def ohlc_project(estimate, hidden):
    """Move only imputed price cells so that each day satisfies low <= open, close <= high."""
    out = estimate.copy()
    o, h, l, c = 0, 1, 2, 3
    for t in np.flatnonzero(hidden[:, :4].any(axis=1)):
        hid = hidden[t, :4]
        v = out[t, :4].copy()
        for _ in range(2):
            if hid[h]:
                v[h] = max(v[h], v[o], v[c], v[l] if not hid[l] else -np.inf)
            if hid[l]:
                v[l] = min(v[l], v[o], v[c], v[h] if not hid[h] else np.inf)
            if hid[o]:
                v[o] = min(max(v[o], v[l]), v[h])
            if hid[c]:
                v[c] = min(max(v[c], v[l]), v[h])
        out[t, :4] = v
    return out


# ----------------------------------------------------------------------------- one condition
def run_condition(parts, ticker, fold, block, rate, seed, mask_type, fixed=None):
    """All methods on one condition. `fixed` overrides CFRI_FIXED = (alpha, gain, volume_rule)."""
    t0 = time.perf_counter()
    masks = {name: condition_mask(values.shape, ticker, fold, name, block, rate, seed, mask_type)
             for name, values in parts.items()}
    train, val, test = (parts[n] for n in ("train", "val", "test"))
    train_mask, val_mask, test_mask = (masks[n] for n in ("train", "val", "test"))
    means, scales = observed_scaler(train, train_mask)
    tr_scaled = corrupt((train - means) / scales, train_mask)
    va_scaled = corrupt((val - means) / scales, val_mask)
    te_scaled = corrupt((test - means) / scales, test_mask)
    linear_val = linear_fill(corrupt(val, val_mask))
    linear_test = linear_fill(corrupt(test, test_mask))
    itri = OriginalITRI().fit(tr_scaled)
    itri_val = itri.transform(va_scaled) * scales + means
    itri_test = itri.transform(te_scaled) * scales + means
    chained = IterativeImputer(estimator=Ridge(alpha=1.0), max_iter=10, tol=1e-3, initial_strategy="mean",
                               imputation_order="ascending", sample_posterior=False, random_state=seed)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        chained.fit(tr_scaled)
        chained_test = chained.transform(te_scaled) * scales + means
    chained_test[~test_mask] = test[~test_mask]

    # CFRI price candidates: validation selection (as in the exploratory run) with full logging
    training = prepare_price_training(train, train_mask, ticker, fold, block, rate, seed)
    val_base, val_designs = prepare_price_eval(val, val_mask)
    test_base, test_designs = prepare_price_eval(test, test_mask)
    log = {}
    best_price = (float("inf"), None, None)
    models_by_alpha = {}
    for alpha in RIDGE_ALPHAS:
        models = fit_price_models(training, alpha)
        models_by_alpha[alpha] = models
        for gain in PRICE_GAINS:
            cand_val = val.copy()
            cand_val[:, :4] = price_candidate(val_base, val_designs, models, gain)
            err = float(np.mean([feature_rmse(val, cand_val, val_mask, scales, d) for d in range(4)]))
            log[f"val_price_a{alpha:g}_g{gain:g}"] = err
            cand_test = test.copy()
            cand_test[:, :4] = price_candidate(test_base, test_designs, models, gain)
            for d in range(4):
                log[f"test_{FEATURES[d]}_a{alpha:g}_g{gain:g}"] = feature_rmse(test, cand_test, test_mask, scales, d)
            if err < best_price[0] - 1e-12:
                best_price = (err, alpha, gain)
    price_alpha, price_gain = best_price[1:]
    # refit (deterministic) exactly as the exploratory run did
    chosen_models = fit_price_models(training, price_alpha)
    selected_prices = price_candidate(test_base, test_designs, chosen_models, price_gain)
    uncrossed_prices = price_candidate(test_base, test_designs, chosen_models, 0.0)

    volume_val = {"linear_raw": linear_val[:, 4], "itri_w1_p5": itri_val[:, 4]}
    volume_test = {"linear_raw": linear_test[:, 4], "itri_w1_p5": itri_test[:, 4]}
    for eta in VOLUME_SHRINK:
        volume_val[f"log_local_{eta:.2f}"] = volume_log_candidate(val, val_mask, eta)
        volume_test[f"log_local_{eta:.2f}"] = volume_log_candidate(test, test_mask, eta)
    volume_errors = {name: float(np.sqrt(np.mean((series[val_mask[:, 4]] - val[val_mask[:, 4], 4]) ** 2)) / scales[4])
                     for name, series in volume_val.items()}
    for name in VOLUME_CANDIDATES:
        log[f"val_volume_{name}"] = volume_errors[name]
        sel = test_mask[:, 4]
        log[f"test_volume_{name}"] = float(np.sqrt(np.mean((volume_test[name][sel] - test[sel, 4]) ** 2)) / scales[4])
    volume_choice = min(volume_errors, key=lambda name: (volume_errors[name], name))

    log_linear = test.copy()
    log_linear[:, :4] = uncrossed_prices
    log_linear[:, 4] = volume_log_candidate(test, test_mask, 0.0)
    adaptive_no_cross = test.copy()
    adaptive_no_cross[:, :4] = uncrossed_prices
    adaptive_no_cross[:, 4] = volume_test[volume_choice]
    adaptive = test.copy()
    adaptive[:, :4] = selected_prices
    adaptive[:, 4] = volume_test[volume_choice]
    f_alpha, f_gain, f_volume = CFRI_FIXED if fixed is None else fixed
    fixed_est = test.copy()
    fixed_est[:, :4] = price_candidate(test_base, test_designs, models_by_alpha[f_alpha], f_gain)
    fixed_est[:, 4] = volume_test[f_volume]
    fixed_no_cross = test.copy()
    fixed_no_cross[:, :4] = uncrossed_prices
    fixed_no_cross[:, 4] = volume_test[f_volume]
    estimates = {"linear_raw": linear_test, "log_linear": log_linear, "itri_w1_p5": itri_test,
                 "chained_ridge": chained_test, "cfri_adaptive_no_cross": adaptive_no_cross,
                 "cfri_adaptive": adaptive, "cfri_fixed_no_cross": fixed_no_cross, "cfri_fixed": fixed_est}
    for value in estimates.values():
        value[~test_mask] = test[~test_mask]
    for name in list(estimates):                      # OHLC projection of imputed cells, for every method
        estimates[name + "_proj"] = ohlc_project(estimates[name], test_mask)
    scores = {m: score_method(test, est, test_mask, scales) for m, est in estimates.items()}
    violations = {m: ohlc_violations(est, test_mask)[0] for m, est in estimates.items()}
    truth_violation, price_rows = ohlc_violations(test, test_mask)
    return {
        "ticker": ticker, "fold": fold, "block_length": block, "rate": rate, "seed": seed, "mask_type": mask_type,
        "mask_sha256": {n: hashlib.sha256(m.tobytes()).hexdigest() for n, m in masks.items()},
        "masked_cells": {n: int(m.sum()) for n, m in masks.items()},
        "price_alpha": price_alpha, "price_gain": price_gain, "volume_choice": volume_choice,
        "scores": scores, "ohlc_violation_share": violations, "truth_ohlc_violation_share": truth_violation,
        "imputed_price_rows": price_rows, "candidates": log, "elapsed_seconds": time.perf_counter() - t0,
    }
'''

FILES['confirm_data.py'] = r'''"""Confirmatory study, step 1: acquire daily OHLCV for the 34 ETFs (2015-01-01 to 2026-09-30) and apply the
score-blind quality rules of the external validation protocol v1 (external_validation/code/acquire_and_qc.py),
extended to the 2026 partial year.

Rules (unchanged from protocol v1 except for the end date): Yahoo Finance through yfinance 1.7.0 with
auto_adjust=True and actions=False; all five OHLCV columns; unique, strictly increasing dates inside the
requested interval; finite positive prices and finite nonnegative volume; valid OHLC order (relative tolerance
1e-6); at least 95% of the XNYS sessions of every calendar year 2015-2025 and of 2026-01-01..2026-09-30; at most
five consecutive absent XNYS sessions; no rows outside the XNYS calendar. An asset that fails is excluded and
logged, never replaced. No imputation method is run and no score is read at this step.

Raw files stay in the user's private Google Drive (never in the public repository); only hashes and QC results
are exported.
"""
from __future__ import annotations

import hashlib
import json
import os
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import cfri_core as C

START, END_EXCLUSIVE, LAST_SESSION = "2015-01-01", "2026-10-01", "2026-09-30"
YEARS = tuple(range(2015, 2027))          # 2026 is the partial year up to LAST_SESSION
COLUMNS = C.FEATURES
RAW_DIR = "raw_do_not_publish"
RECEIPT = "data_receipt.json"


def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()


def _atomic(path: Path, data: bytes) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_bytes(data)
    os.replace(tmp, path)


def max_true_run(flags) -> int:  # as in acquire_and_qc.py
    flags = np.asarray(flags, dtype=bool)
    if flags.size == 0:
        return 0
    edges = np.flatnonzero(np.diff(np.r_[False, flags, False]))
    lengths = edges[1::2] - edges[::2]
    return int(lengths.max()) if lengths.size else 0


def normalized_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:  # as in acquire_and_qc.py, new end date
    if frame.empty:
        raise ValueError("empty provider response")
    out = frame.copy()
    if isinstance(out.columns, pd.MultiIndex):
        for level in range(out.columns.nlevels):
            names = [str(n).lower().replace(" ", "_") for n in out.columns.get_level_values(level)]
            if all(n in names for n in COLUMNS):
                out.columns = names
                break
        else:
            raise ValueError("cannot locate OHLCV level in provider columns")
    else:
        out.columns = [str(n).lower().replace(" ", "_") for n in out.columns]
    missing = set(COLUMNS).difference(out.columns)
    if missing:
        raise ValueError(f"missing provider columns: {sorted(missing)}")
    out = out.loc[:, list(COLUMNS)]
    if not isinstance(out.index, pd.DatetimeIndex):
        out.index = pd.to_datetime(out.index)
    if out.index.tz is not None:
        out.index = out.index.tz_localize(None)
    out.index = out.index.normalize()
    if out.index.has_duplicates:
        raise ValueError("duplicate trading dates")
    if not out.index.is_monotonic_increasing:
        raise ValueError("trading dates not strictly increasing")
    if (out.index < pd.Timestamp(START)).any() or (out.index >= pd.Timestamp(END_EXCLUSIVE)).any():
        raise ValueError("provider returned dates outside requested interval")
    return out.astype(float)


def xnys_sessions() -> pd.DatetimeIndex:
    import exchange_calendars as xcals
    s = pd.DatetimeIndex(xcals.get_calendar("XNYS").sessions_in_range(START, LAST_SESSION))
    if s.tz is not None:
        s = s.tz_localize(None)
    return s.normalize()


def assess_quality(frame: pd.DataFrame, sessions: pd.DatetimeIndex) -> dict:  # as in acquire_and_qc.py
    reasons = []
    values = frame.loc[:, list(COLUMNS)].to_numpy(float)
    if not np.isfinite(values).all():
        reasons.append("nonfinite_ohlcv")
    if np.isfinite(values).all():
        if (frame.loc[:, ("open", "high", "low", "close")] <= 0).any().any():
            reasons.append("nonpositive_price")
        if (frame["volume"] < 0).any():
            reasons.append("negative_volume")
        tol = 1e-6 * frame.loc[:, ("open", "high", "low", "close")].max(axis=1)
        if ((frame["high"] + tol < frame[["open", "low", "close"]].max(axis=1)) |
                (frame["low"] - tol > frame[["open", "high", "close"]].min(axis=1))).any():
            reasons.append("invalid_ohlc_order")
    observed = pd.DatetimeIndex(frame.index)
    missing = ~sessions.isin(observed)
    non_us = observed.difference(sessions)
    yearly = {}
    for year in YEARS:
        expected = sessions[sessions.year == year]
        present = int(expected.isin(observed).sum())
        coverage = present / len(expected)
        yearly[str(year)] = {"expected_sessions": len(expected), "observed_sessions": present, "coverage": coverage}
        if coverage < 0.95:
            reasons.append(f"coverage_below_95pct_{year}")
    longest = max_true_run(missing)
    if longest > 5:
        reasons.append("more_than_five_consecutive_absent_us_sessions")
    if len(non_us):
        reasons.append("rows_outside_us_exchange_calendar")
    return {"eligible": not reasons, "reasons": sorted(set(reasons)), "row_count": len(frame),
            "first_date": frame.index.min().strftime("%Y-%m-%d"), "last_date": frame.index.max().strftime("%Y-%m-%d"),
            "sessions_2026": int((frame.index.year == 2026).sum()), "longest_absent_us_session_run": longest,
            "non_us_session_count": len(non_us), "yearly_coverage": yearly}


def methods_started(root: Path) -> bool:
    return any(p.is_file() for d in ("fold_D_cpu", "fold_D_saits") if (root / d).is_dir() for p in (root / d).iterdir())


def provider_failures(receipt: dict) -> list:
    return [a["ticker"] for a in receipt["assets"]
            if not a["eligible"] and any(r.startswith("acquisition_or_qc_error") for r in a["reasons"])]


def _download(yf, ticker: str, attempts: int = 3) -> pd.DataFrame:
    """Up to three attempts (pauses of 10 and 30 seconds; CFRI_RETRY_PAUSE_SCALE rescales them for offline tests)."""
    scale = float(os.environ.get("CFRI_RETRY_PAUSE_SCALE", "1"))
    for attempt in range(attempts):
        try:
            got = yf.download(ticker, start=START, end=END_EXCLUSIVE, auto_adjust=True, actions=False,
                              progress=False, threads=False)
            return normalized_ohlcv(got)
        except Exception:
            if attempt == attempts - 1:
                raise
            time.sleep(scale * (10, 30)[attempt])


def acquire(root: Path) -> dict:
    """Download (or reuse the Drive cache), check, and write data_receipt.json.

    Once any method output exists the receipt is final and is returned unchanged. Before that, a repeated call
    re-attempts only assets that failed with a download or provider error (they have no cached file); cached files
    are reused with their original retrieval record, so every other QC decision is reproduced exactly."""
    import exchange_calendars as xcals
    import yfinance as yf
    if (root / RECEIPT).exists():
        old = json.loads((root / RECEIPT).read_text())
        if methods_started(root) or not provider_failures(old) or old.get("provider_failures_accepted"):
            print("data_receipt.json exists and is final:", old["eligible_count"], "eligible assets", flush=True)
            return old
        print("retrying provider failures before any method has run:", provider_failures(old), flush=True)
    raw = root / RAW_DIR
    raw.mkdir(parents=True, exist_ok=True)
    sessions = xnys_sessions()
    receipt = {"study": "CFRI confirmatory fold D", "start_inclusive": START, "end_exclusive": END_EXCLUSIVE,
               "provider": "Yahoo Finance via yfinance; auto_adjust=True; actions=False",
               "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                               "yfinance": yf.__version__, "exchange_calendars": xcals.__version__},
               "code_id": C.CODE_ID, "xnys_sessions_2026_to_cutoff": int((sessions.year == 2026).sum()),
               "acquisition_started_utc": datetime.now(timezone.utc).isoformat(), "assets": []}
    for i, (ticker, country, region) in enumerate(C.ASSETS, 1):
        path = raw / f"{ticker}_{START}_{END_EXCLUSIVE}.csv"
        meta_path = path.with_suffix(".meta.json")
        rec = {"ticker": ticker, "country": country, "region": region, "sequence": i}
        try:
            if path.exists():
                frame = pd.read_csv(path, index_col=0, parse_dates=True, float_precision="round_trip")
                meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
                rec["source"] = "drive_cache"
                rec["retrieved_utc"] = meta.get("retrieved_utc")
            else:
                frame = _download(yf, ticker)
                retrieved = datetime.now(timezone.utc).isoformat()
                payload = frame.to_csv(float_format="%.17g", index_label="date").encode()
                _atomic(path, payload)
                _atomic(meta_path, json.dumps({"ticker": ticker, "retrieved_utc": retrieved,
                                               "yfinance": yf.__version__, "sha256": sha256_bytes(payload)}).encode())
                rec["source"] = "fresh_yfinance_download"
                rec["retrieved_utc"] = retrieved
            rec["sha256"] = sha256_bytes(path.read_bytes())
            frame = normalized_ohlcv(frame)
            rec.update(assess_quality(frame, sessions))
        except Exception as exc:  # logged exclusion, never a replacement
            rec.update({"eligible": False, "reasons": [f"acquisition_or_qc_error:{type(exc).__name__}:{exc}"]})
        receipt["assets"].append(rec)
        print(f"[{i:02d}/34] {ticker}: {'eligible' if rec['eligible'] else 'EXCLUDED ' + str(rec['reasons'])}",
              flush=True)
    receipt["acquisition_finished_utc"] = datetime.now(timezone.utc).isoformat()
    receipt["eligible_count"] = sum(a["eligible"] for a in receipt["assets"])
    receipt["excluded"] = [{"ticker": a["ticker"], "reasons": a["reasons"]} for a in receipt["assets"] if not a["eligible"]]
    receipt["provider_failures_accepted"] = False
    _atomic(root / RECEIPT, json.dumps(receipt, ensure_ascii=False, indent=2).encode())
    return receipt


def accept_provider_failures(root: Path) -> dict:
    """Record that repeated provider failures are accepted as exclusions (score-blind; before any method runs)."""
    receipt = json.loads((root / RECEIPT).read_text())
    if methods_started(root):
        raise RuntimeError("methods have already run; the receipt is final")
    receipt["provider_failures_accepted"] = True
    receipt["provider_failures_accepted_utc"] = datetime.now(timezone.utc).isoformat()
    _atomic(root / RECEIPT, json.dumps(receipt, ensure_ascii=False, indent=2).encode())
    return receipt


def load_frames(root: Path) -> dict:
    """Eligible assets from data_receipt.json, each CSV verified against its recorded SHA-256."""
    receipt = json.loads((root / RECEIPT).read_text())
    frames = {}
    for rec in receipt["assets"]:
        if not rec["eligible"]:
            continue
        path = root / RAW_DIR / f"{rec['ticker']}_{START}_{END_EXCLUSIVE}.csv"
        blob = path.read_bytes()
        if sha256_bytes(blob) != rec["sha256"]:
            raise ValueError(f"{rec['ticker']}: data file changed after the receipt was written")
        frame = pd.read_csv(path, index_col=0, parse_dates=True, float_precision="round_trip")
        frames[rec["ticker"]] = normalized_ohlcv(frame)
    return frames
'''

FILES['confirm_cpu.py'] = r'''"""Confirmatory study, step 2: CPU methods on fold D, feature-wise and whole-row masks.

Every condition runs cfri_core.run_condition, which scores all methods on identical masks: linear_raw,
log_linear, itri_w1_p5, chained_ridge, cfri_adaptive(_no_cross), cfri_fixed(_no_cross), and each of them
followed by the OHLC projection of imputed cells (suffix _proj). CFRI as evaluated in the protocol is
cfri_fixed_proj with the setting C.CFRI_FIXED frozen from folds A-C validation data.

One JSON-lines shard per (ticker, mask type) with a SHA-256 sidecar; finished shards are skipped on restart.
Infeasible conditions (C.feasible) are listed in the sidecar. A condition that raises an error is written as an
error line; the analysis excludes it for every method and reports it.
"""
from __future__ import annotations

import hashlib
import json
import os
from multiprocessing import Pool
from pathlib import Path

import numpy as np

import cfri_core as C

FOLD = "D"
OUT = "fold_D_cpu"


def _jsonable(x):
    if isinstance(x, dict):
        return {k: _jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonable(v) for v in x]
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.integer):
        return int(x)
    return x


def code_digest() -> str:
    """SHA-256 over the code that determines the CPU results (this file and cfri_core.py)."""
    h = hashlib.sha256()
    for path in (Path(C.__file__), Path(__file__)):
        h.update(path.read_bytes())
    return h.hexdigest()


def _atomic(path: Path, data: bytes) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_bytes(data)
    os.replace(tmp, path)


def shard_complete(path: Path, n_conditions: int | None = None) -> bool:
    """True when the shard and its sidecar agree (SHA-256, code digest and, if given, the condition count)."""
    side = path.with_name(path.name + ".done.json")
    if not (path.exists() and side.exists()):
        return False
    try:
        info = json.loads(side.read_text())
    except (OSError, ValueError):
        return False
    return (info.get("sha256") == hashlib.sha256(path.read_bytes()).hexdigest() and info.get("code") == C.CODE_ID
            and info.get("code_sha256") == code_digest()
            and (n_conditions is None or info.get("conditions") == n_conditions))


def conditions_for(parts, mask_type):
    rows = {k: len(v) for k, v in parts.items()}
    keep, dropped = [], []
    for block in C.BLOCKS:
        for rate in C.RATES:
            ok = C.feasible(rows, block, rate, mask_type)
            for seed in C.SEEDS:
                (keep if ok else dropped).append((block, rate, seed))
    return keep, dropped


def _shard(job):
    root, ticker, mask_type, frame = job
    out = Path(root) / OUT
    path = out / f"{ticker}_{mask_type}.jsonl"
    side = path.with_name(path.name + ".done.json")
    parts = C.split_frame(frame, FOLD)
    keep, dropped = conditions_for(parts, mask_type)
    if shard_complete(path, len(keep)):
        return ticker, mask_type, "skip", 0
    lines, errors = [], 0
    for b, r, s in keep:
        try:
            res = C.run_condition(parts, ticker, FOLD, b, r, s, mask_type)
        except Exception as exc:  # reported, excluded for all methods in the analysis
            res = {"ticker": ticker, "fold": FOLD, "block_length": b, "rate": r, "seed": s, "mask_type": mask_type,
                   "error": f"{type(exc).__name__}: {exc}"}
            errors += 1
        lines.append(json.dumps(_jsonable(res), sort_keys=True, allow_nan=False))
    data = ("\n".join(lines) + "\n").encode()
    _atomic(path, data)
    _atomic(side, json.dumps({"sha256": hashlib.sha256(data).hexdigest(), "code": C.CODE_ID,
                              "code_sha256": code_digest(), "conditions": len(keep), "errors": errors,
                              "rows_by_partition": {k: len(v) for k, v in parts.items()},
                              "dropped_infeasible": dropped}).encode())
    return ticker, mask_type, "done", errors


def run_cpu(root: Path, frames: dict, mask_types=C.MASK_TYPES, workers: int | None = None) -> dict:
    (root / OUT).mkdir(parents=True, exist_ok=True)
    jobs = [(str(root), t, m, f) for m in mask_types for t, f in frames.items()]
    workers = workers or os.cpu_count() or 1
    status = {}
    with Pool(workers) as pool:
        for i, (ticker, mask_type, state, errors) in enumerate(pool.imap_unordered(_shard, jobs), 1):
            status[f"{ticker}_{mask_type}"] = {"state": state, "errors": errors}
            print(f"[CPU {i}/{len(jobs)}] {ticker} {mask_type}: {state}" + (f", {errors} errors" if errors else ""),
                  flush=True)
    complete = all(shard_complete(root / OUT / f"{t}_{m}.jsonl") for m in mask_types for t in frames)
    import platform, scipy, sklearn, pandas
    log = {"code": C.CODE_ID, "code_sha256": code_digest(), "workers": workers, "shards": len(jobs),
           "complete": complete, "status": status,
           "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pandas.__version__,
                           "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "cpu_count": os.cpu_count()}}
    _atomic(root / OUT / "cpu_run_log.json", json.dumps(log, indent=1).encode())
    return log
'''

FILES['confirm_saits.py'] = r'''"""Confirmatory study, step 3: normalized SAITS on fold D (GPU), protocol amendment v2 settings.

Same windows, architecture, budgets and per-window instance normalization as the archived amendment v2 run
(external_validation/code/run_neural_windownorm.py): 64-day windows, stride 16, batch 32; SAITS with 2 layers,
d_model 64, 4 heads, d_k = d_v = 16, d_ffn 128, dropout 0.1; V1 = 30 epochs (patience 5), V2 = 200 epochs
(patience 20); each window standardized by the mean and SD of its own observed cells, predictions
de-standardized with the same statistics, observed cells restored. Masks and scaling come from cfri_core, so
SAITS sees exactly the conditions of the CPU methods. Each condition is scored raw (saits_winnorm_V*) and after
the same OHLC projection as the CPU methods (saits_winnorm_V*_proj).

One CSV shard per (ticker, mask type, variant) with a SHA-256 sidecar; finished shards are skipped on restart.
"""
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

import cfri_core as C

FOLD = "D"
OUT = "fold_D_saits"
N_STEPS, STRIDE, BATCH_SIZE = 64, 16, 32
SAITS_CONFIG = {"n_layers": 2, "d_model": 64, "n_heads": 4, "d_k": 16, "d_v": 16, "d_ffn": 128, "dropout": 0.1}
VARIANTS = {"V1": dict(epochs=30, patience=5), "V2": dict(epochs=200, patience=20)}


def window_starts(length, n_steps=N_STEPS, stride=STRIDE):
    if length < n_steps:
        raise ValueError("partition shorter than the neural window")
    starts = list(range(0, length - n_steps + 1, stride))
    if starts[-1] != length - n_steps:
        starts.append(length - n_steps)
    return starts


def as_windows(values, starts):
    return np.stack([values[s:s + N_STEPS] for s in starts]).astype(np.float32, copy=False)


def fold_windows(predictions, starts, observed):
    sums = np.zeros(observed.shape, dtype=np.float64)
    counts = np.zeros(observed.shape[0], dtype=np.int32)
    for p, s in zip(predictions, starts):
        sums[s:s + N_STEPS] += p
        counts[s:s + N_STEPS] += 1
    if np.any(counts == 0):
        raise AssertionError("window coverage incomplete")
    out = sums / counts[:, None]
    seen = np.isfinite(observed)
    out[seen] = observed[seen]
    if not np.isfinite(out).all():
        raise ValueError("non-finite neural estimates")
    return out


def window_stats(win):
    mu = np.nanmean(win, axis=1, keepdims=True)
    sd = np.nanstd(win, axis=1, keepdims=True)
    n_obs = np.sum(np.isfinite(win), axis=1, keepdims=True)
    bad = (n_obs < 2) | ~np.isfinite(sd) | (sd < 1e-6)
    return np.where(bad, 0.0, mu).astype(np.float32), np.where(bad, 1.0, sd).astype(np.float32), int(bad.sum())


def set_training_seed(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def training_seed(ticker, block, rate, rep, mask_type):
    key = f"{C.ORIGINAL_MANIFEST_SHA}|{ticker}|{FOLD}|{block}|{rate:.2f}|{rep}|saits|{mask_type}|confirmatory-v1"
    return int.from_bytes(hashlib.sha256(key.encode()).digest()[:4], "big")


def fit_predict(train_in, val_in, val_truth, test_in, seed, epochs, patience, device):
    import warnings
    from pypots.imputation import SAITS
    set_training_seed(seed)
    tr_s, va_s, te_s = window_starts(len(train_in)), window_starts(len(val_in)), window_starts(len(test_in))
    Xtr, Xva, Xva_ori, Xte = (as_windows(a, s) for a, s in ((train_in, tr_s), (val_in, va_s), (val_truth, va_s), (test_in, te_s)))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)   # all-missing window rows are handled by the fallback
        mtr, s_tr, f1 = window_stats(Xtr)
        mva, s_va, f2 = window_stats(Xva)
        mte, s_te, f3 = window_stats(Xte)
    Xtr = (Xtr - mtr) / s_tr
    Xva_ori = (Xva_ori - mva) / s_va
    Xva = (Xva - mva) / s_va
    Xte = (Xte - mte) / s_te
    model = SAITS(**SAITS_CONFIG, n_steps=N_STEPS, n_features=len(C.FEATURES), batch_size=BATCH_SIZE, epochs=epochs,
                  patience=min(patience, epochs), device=device, model_saving_strategy=None, verbose=False)
    t0 = time.perf_counter()
    model.fit({"X": Xtr}, {"X": Xva, "X_ori": Xva_ori})
    Pte = np.asarray(model.predict({"X": Xte})["imputation"]) * s_te + mte
    elapsed = time.perf_counter() - t0
    del model
    return fold_windows(Pte, te_s, test_in), elapsed, f1 + f2 + f3


def _environment(device):
    import torch
    import pypots
    return {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "pypots": pypots.__version__, "device": device,
            "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
            "cudnn": torch.backends.cudnn.version() if device == "cuda" else None}


MAX_CONSECUTIVE_FAILURES = 3


def code_digest() -> str:
    """SHA-256 over the code that determines SAITS results (this file and cfri_core.py)."""
    h = hashlib.sha256()
    for path in (Path(C.__file__), Path(__file__)):
        h.update(path.read_bytes())
    return h.hexdigest()


def job_fingerprint(variant: str, mask_type: str) -> str:
    """Identifies the code, settings and job of a SAITS shard; finished shards are reused only when it matches."""
    return hashlib.sha256(json.dumps({"code": C.CODE_ID, "code_sha256": code_digest(), "variant": variant,
                                      "cfg": VARIANTS[variant], "mask": mask_type, "saits": SAITS_CONFIG,
                                      "win": [N_STEPS, STRIDE, BATCH_SIZE]}, sort_keys=True).encode()).hexdigest()


def _atomic(path: Path, data: bytes) -> None:
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_bytes(data)
    os.replace(tmp, path)


def _read_json(path: Path):
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return None


def _is_device_error(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(k in text for k in ("cuda", "cudnn", "cublas", "device-side", "accelerator", "out of memory"))


def run_saits(root: Path, frames: dict, variant: str, mask_type: str, smoke_conditions: int | None = None,
              threads: int = 2):
    """Train and score SAITS for every feasible condition. A condition that fails for a model-level reason is
    recorded and skipped; device errors and three consecutive failures stop the run (resume later). A shard
    without any completed condition is never marked as finished."""
    import logging
    import torch
    torch.set_num_threads(threads)
    try:   # silence the per-model "saving_path not given" notice, as in the archived neural runs
        from pypots.utils.logging import logger as pypots_logger
        pypots_logger.addFilter(lambda record: "saving_path not given" not in record.getMessage())
    except Exception:
        pass
    device = "cuda" if torch.cuda.is_available() else "cpu"
    cfg = VARIANTS[variant]
    out = root / OUT
    out.mkdir(parents=True, exist_ok=True)
    fingerprint = job_fingerprint(variant, mask_type)
    if not smoke_conditions:
        meta_path = out / f"run_metadata_{variant}_{mask_type}.json"
        history = (_read_json(meta_path) or {}).get("sessions", [])
        history.append({"started_utc": pd.Timestamp.now(tz="UTC").isoformat(), **_environment(device)})
        _atomic(meta_path, json.dumps({"variant": variant, "mask_type": mask_type, "config": cfg, "saits": SAITS_CONFIG,
                                       "fingerprint": fingerprint, "sessions": history}, indent=2).encode())
    t_start, n_done, consecutive = time.time(), 0, 0
    for i, (ticker, frame) in enumerate(frames.items(), 1):
        stem = f"{ticker}_{mask_type}_saits_{variant}"
        csv_path, side = out / f"{stem}.csv", out / f"{stem}.done.json"
        if not smoke_conditions and csv_path.exists() and side.exists():
            info = _read_json(side) or {}
            if info.get("fingerprint") == fingerprint and info.get("sha256") == hashlib.sha256(csv_path.read_bytes()).hexdigest():
                continue
        parts = C.split_frame(frame, FOLD)
        rows_by_part = {k: len(v) for k, v in parts.items()}
        train, val, test = parts["train"], parts["val"], parts["test"]
        rows, errors = [], []
        for block in C.BLOCKS:
            for rate in C.RATES:
                if not C.feasible(rows_by_part, block, rate, mask_type):
                    continue
                for rep in C.SEEDS:
                    m = {n: C.condition_mask(v.shape, ticker, FOLD, n, block, rate, rep, mask_type) for n, v in parts.items()}
                    mean, scale = C.observed_scaler(train, m["train"])
                    train_in = C.corrupt((train - mean) / scale, m["train"])
                    val_truth = (val - mean) / scale
                    val_in = C.corrupt(val_truth, m["val"])
                    test_in = C.corrupt((test - mean) / scale, m["test"])
                    seed = training_seed(ticker, block, rate, rep, mask_type)
                    try:
                        est, elapsed, fallbacks = fit_predict(train_in, val_in, val_truth, test_in, seed, device=device, **cfg)
                    except Exception as exc:
                        if _is_device_error(exc):
                            raise
                        consecutive += 1
                        errors.append({"ticker": ticker, "variant": variant, "mask_type": mask_type, "block_length": block,
                                       "rate": rate, "seed": rep, "error": f"{type(exc).__name__}: {exc}"})
                        print(f"SAITS {variant} {mask_type} {ticker} b{block} r{rate} s{rep} failed: {errors[-1]['error']}",
                              flush=True)
                        if consecutive >= MAX_CONSECUTIVE_FAILURES:
                            raise RuntimeError(f"{consecutive} consecutive SAITS failures; stopping (resume later)") from exc
                        continue
                    consecutive = 0
                    est_raw = est * scale + mean
                    est_raw[~m["test"]] = test[~m["test"]]
                    for name, e in ((f"saits_winnorm_{variant}", est_raw),
                                    (f"saits_winnorm_{variant}_proj", C.ohlc_project(est_raw, m["test"]))):
                        scores = C.score_method(test, e, m["test"], scale)
                        viol, _ = C.ohlc_violations(e, m["test"])
                        for feat, (rmse, mae, n) in scores.items():
                            rows.append({"ticker": ticker, "fold": FOLD, "block_length": block, "rate": rate, "seed": rep,
                                         "mask_type": mask_type, "method": name, "feature": feat, "rmse": rmse,
                                         "mae": mae, "masked_cells": n, "model_seed": seed, "seconds": elapsed,
                                         "window_fallbacks": fallbacks, "ohlc_violation_share": viol})
                    n_done += 1
                    if smoke_conditions and n_done >= smoke_conditions:
                        print(f"SAITS {variant} {mask_type} smoke test passed: {n_done} condition(s), "
                              f"{elapsed:.1f} s for the last one on {device}", flush=True)
                        return pd.DataFrame(rows)
        if not rows:
            raise RuntimeError(f"SAITS {variant} {mask_type} {ticker}: no condition completed; shard not saved")
        data = pd.DataFrame(rows).to_csv(index=False, float_format="%.17g").encode()
        _atomic(csv_path, data)
        _atomic(side, json.dumps({"fingerprint": fingerprint, "sha256": hashlib.sha256(data).hexdigest(),
                                  "rows": len(rows), "errors": errors}).encode())
        el = time.time() - t_start
        print(f"[SAITS {variant} {mask_type} {i}/{len(frames)}] {ticker} saved, {len(errors)} errors | "
              f"{el / 60:.1f} min this session", flush=True)
    return None
'''

FILES['confirm_analyze.py'] = r'''"""Confirmatory study, step 4: prespecified analysis of fold D and export of a results ZIP (no raw prices).

Endpoint: for each condition, the mean over the five features of RMSE on masked test cells divided by the
training-observed SD of the feature (NRMSE); per asset, the mean over its conditions; per method, the mean (and
median) over assets.

Primary hypotheses (feature-wise masks), CFRI = cfri_fixed_proj:
  H1 CFRI vs log-linear interpolation (log_linear);
  H2 CFRI vs original ITRI, w = 1, p = 5 (itri_w1_p5);
  H3 CFRI vs CFRI without the cross-feature residual (cfri_fixed_no_cross_proj; same volume rule and projection);
  H4 CFRI vs normalized SAITS V2 (saits_winnorm_V2, protocol amendment v2 settings), paired on the conditions
     SAITS completed.
Comparators of H1, H2 and H4 are used exactly as defined in the earlier protocols (no OHLC projection); the
projection-equalized comparisons (log_linear_proj, itri_w1_p5_proj, saits_winnorm_V2_proj) are secondary.
Each hypothesis is met when CFRI has the lower mean asset-level NRMSE AND is lower in at least k of the n analyzed
assets, k being the smallest count with one-sided binomial tail P(X >= k | n, 1/2) < 0.01 (k = 25 for n = 34).
Assets are not independent; k is a prespecified consistency threshold, not a significance test. Everything else
is descriptive: medians, paired differences with bootstrap intervals over assets, Wilcoxon signed-rank p-values,
block/rate/feature/region breakdowns, leave-one-region-out, MAE, OHLC violations, whole-row masks, SAITS V1.

The same functions summarize the exploratory folds A-C (analyze_exploratory.py).
"""
from __future__ import annotations

import glob
import hashlib
import json
import math
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

import cfri_core as C
import confirm_cpu
import confirm_saits

KEY = ["ticker", "fold", "block_length", "rate", "seed"]
PRIMARY = C.PRIMARY_METHOD
HYPOTHESES = (("H1", "log_linear"), ("H2", "itri_w1_p5"), ("H3", "cfri_fixed_no_cross_proj"))
SAITS_HYPOTHESIS = ("H4", "saits_winnorm_V2")
PROJECTION_EQUALIZED = ("log_linear_proj", "itri_w1_p5_proj", "saits_winnorm_V2_proj")
REGION = {t: r for t, _, r in C.ASSETS}
BOOT_SEED, N_BOOT = 20261027, 10000
CPU_ORDER = list(C.METHODS)
SAITS_METHODS = ("saits_winnorm_V1", "saits_winnorm_V2", "saits_winnorm_V1_proj", "saits_winnorm_V2_proj")


def threshold(n: int, alpha: float = 0.01) -> int:
    for k in range(n + 1):
        if sum(math.comb(n, j) for j in range(k, n + 1)) / 2 ** n < alpha:
            return k
    return n + 1


def condition_level(long: pd.DataFrame) -> pd.DataFrame:
    """Feature rows -> one row per condition and method (macro NRMSE, macro NMAE, price-only, volume)."""
    g = long.groupby(KEY + ["mask_type", "method"])
    out = g.agg(nrmse=("rmse", "mean"), nmae=("mae", "mean"), n_features=("rmse", "size")).reset_index()
    price = long[long.feature != "volume"].groupby(KEY + ["mask_type", "method"]).rmse.mean().rename("nrmse_price")
    vol = long[long.feature == "volume"].groupby(KEY + ["mask_type", "method"]).rmse.mean().rename("nrmse_volume")
    out = out.merge(price.reset_index(), on=KEY + ["mask_type", "method"]).merge(vol.reset_index(), on=KEY + ["mask_type", "method"])
    if (out.n_features != len(C.FEATURES)).any():
        raise ValueError("a condition lacks one of the five features")
    return out.drop(columns="n_features")


def asset_table(cond: pd.DataFrame, value: str = "nrmse") -> pd.DataFrame:
    return cond.groupby(["ticker", "method"])[value].mean().unstack()


def paired(asset: pd.DataFrame, a: str, b: str) -> dict:
    d = (asset[a] - asset[b]).dropna()
    n = len(d)
    if n == 0:
        return {"method": a, "comparator": b, "assets": 0, "criterion_met": False}
    rng = np.random.default_rng(BOOT_SEED)
    boots = d.to_numpy()[rng.integers(0, n, size=(N_BOOT, n))].mean(axis=1)
    try:
        from scipy.stats import wilcoxon
        p = float(wilcoxon(d.to_numpy()).pvalue) if n > 0 and (d != 0).any() else float("nan")
    except Exception:
        p = float("nan")
    k = threshold(n)
    lower_mean = bool(asset.loc[d.index, a].mean() < asset.loc[d.index, b].mean())
    wins = int((d < 0).sum())
    return {"method": a, "comparator": b, "assets": n, "mean_method": float(asset.loc[d.index, a].mean()),
            "mean_comparator": float(asset.loc[d.index, b].mean()), "median_method": float(asset.loc[d.index, a].median()),
            "median_comparator": float(asset.loc[d.index, b].median()), "mean_difference": float(d.mean()),
            "relative_difference_pct": float(100 * d.mean() / asset.loc[d.index, b].mean()),
            "median_difference": float(d.median()), "boot95_low": float(np.percentile(boots, 2.5)),
            "boot95_high": float(np.percentile(boots, 97.5)), "wilcoxon_p_descriptive": p,
            "method_lower_in_assets": wins, "threshold_k": k, "criterion_met": bool(lower_mean and wins >= k)}


def breakdown(cond: pd.DataFrame, long: pd.DataFrame, methods) -> dict:
    c = cond[cond.method.isin(methods)]
    by = {}
    for col in ("block_length", "rate"):
        by[col] = (c.groupby([col, "ticker", "method"]).nrmse.mean().groupby([col, "method"]).mean()
                   .unstack().round(5).to_dict(orient="index"))
    f = long[long.method.isin(methods)]
    by["feature"] = (f.groupby(["feature", "ticker", "method"]).rmse.mean().groupby(["feature", "method"]).mean()
                     .unstack().round(5).to_dict(orient="index"))
    a = asset_table(c)
    a["region"] = a.index.map(REGION)
    by["region_mean"] = a.groupby("region")[list(methods)].mean().round(5).to_dict(orient="index")
    return by


def summarize(cond: pd.DataFrame, long: pd.DataFrame, viol: pd.DataFrame | None, primary_tests: bool):
    """Summary for one mask type (and one fold or pooled folds) of the CPU methods."""
    present = [m for m in CPU_ORDER if m in set(cond.method)]
    counts = cond.groupby("method").size()
    n_cond = int(counts.max())
    methods = [m for m in present if counts.get(m, 0) == n_cond]        # identical conditions for every method
    asset = asset_table(cond[cond.method.isin(methods)])
    out = {"assets": int(len(asset)), "conditions": n_cond, "methods": methods,
           "mean": asset.mean().round(5).to_dict(), "median": asset.median().round(5).to_dict(),
           "mean_nmae": asset_table(cond[cond.method.isin(methods)], "nmae").mean().round(5).to_dict(),
           "mean_price_nrmse": asset_table(cond[cond.method.isin(methods)], "nrmse_price").mean().round(5).to_dict(),
           "mean_volume_nrmse": asset_table(cond[cond.method.isin(methods)], "nrmse_volume").mean().round(5).to_dict(),
           "paired_vs_cfri": {m: paired(asset, PRIMARY, m) for m in methods if m != PRIMARY},
           "breakdowns": breakdown(cond, long, methods)}
    if viol is not None and len(viol):
        out["ohlc_violation_share"] = viol.groupby("method").ohlc_violation_share.mean().round(5).to_dict()
    if primary_tests:
        out["hypotheses"] = {h: paired(asset, PRIMARY, comp) for h, comp in HYPOTHESES}
        loro = {}
        for region in sorted(set(REGION.values())):
            keep = [t for t in asset.index if REGION[t] != region]
            loro[f"without_{region}"] = {h: {k: v for k, v in paired(asset.loc[keep], PRIMARY, comp).items()
                                             if k in ("assets", "mean_difference", "method_lower_in_assets",
                                                      "threshold_k", "criterion_met")}
                                         for h, comp in HYPOTHESES}
        out["leave_one_region_out"] = loro
    return out, asset


def saits_comparisons(cond: pd.DataFrame) -> dict:
    """CFRI vs each SAITS variant, paired on the conditions SAITS completed."""
    res = {}
    cf = cond[cond.method == PRIMARY].set_index(KEY)["nrmse"]
    for s in SAITS_METHODS:
        sc = cond[cond.method == s].set_index(KEY)["nrmse"]
        if sc.empty:
            continue
        both = pd.concat({PRIMARY: cf, s: sc}, axis=1, join="inner").reset_index()
        asset = both.groupby("ticker")[[PRIMARY, s]].mean()
        res[s] = {"conditions_paired": int(len(both)), "conditions_cfri": int(len(cf)), **paired(asset, PRIMARY, s)}
    return res


# ----------------------------------------------------------------------------- fold D loading
def _read_json(path: Path):
    try:
        return json.loads(path.read_text())
    except (OSError, ValueError):
        return None


def _sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load(root: Path, tickers, mask_types=C.MASK_TYPES):
    """Read verified shards. Every eligible asset must have a complete CPU shard per mask type (otherwise the
    analysis stops); SAITS shards are used when their sidecar verifies, and their coverage is reported."""
    rows, viol, errors, problems, seconds = [], [], [], [], []
    for mt in mask_types:
        for t in tickers:
            f = root / "fold_D_cpu" / f"{t}_{mt}.jsonl"
            if not confirm_cpu.shard_complete(f):     # SHA-256, CODE_ID and code digest must match this code
                problems.append(f.name)
                continue
            for line in open(f):
                r = json.loads(line)
                base = {k: r[k] for k in KEY + ["mask_type"]}
                if "error" in r:
                    errors.append({**base, "error": r["error"]})
                    continue
                for m, feats in r["scores"].items():
                    for feat, (rmse, mae, n) in feats.items():
                        rows.append({**base, "method": m, "feature": feat, "rmse": rmse, "mae": mae})
                for m, v in r["ohlc_violation_share"].items():
                    viol.append({**base, "method": m, "ohlc_violation_share": v})
                viol.append({**base, "method": "truth", "ohlc_violation_share": r["truth_ohlc_violation_share"]})
                seconds.append(r["elapsed_seconds"])
    if problems:
        raise RuntimeError(f"CPU results missing, modified or from other code for {problems}; "
                           "run the cpu step to completion first")
    long = pd.DataFrame(rows)
    saits_errors, sv, coverage = [], [], {}
    for f in sorted((root / "fold_D_saits").glob("*.csv")) if (root / "fold_D_saits").is_dir() else []:
        ticker, mt, _, variant = f.stem.rsplit("_", 3)
        info = _read_json(f.with_name(f.stem + ".done.json"))
        if (ticker not in tickers or info is None or info.get("sha256") != _sha(f) or not info.get("rows")
                or info.get("fingerprint") != confirm_saits.job_fingerprint(variant, mt)):
            continue    # unverified, empty or produced by other code: not used
        job = f"{variant}_{mt}"
        coverage[job] = coverage.get(job, 0) + 1
        saits_errors += info.get("errors", [])
        sv.append(pd.read_csv(f, float_precision="round_trip"))
    run_time = {"cpu_all_methods_seconds_per_condition": float(np.mean(seconds)) if seconds else None}
    if sv:
        s = pd.concat(sv, ignore_index=True)
        per = s.drop_duplicates(KEY + ["mask_type", "method"])
        mean_s = per[~per.method.str.endswith("_proj")].groupby(["method", "mask_type"]).seconds.mean().round(3)
        run_time["saits_seconds_per_condition"] = {f"{m}_{mt}": float(v) for (m, mt), v in mean_s.items()}
        long = pd.concat([long, s[KEY + ["mask_type", "method", "feature", "rmse", "mae"]]], ignore_index=True)
        viol += s.drop_duplicates(KEY + ["mask_type", "method"])[KEY + ["mask_type", "method", "ohlc_violation_share"]].to_dict("records")
    long["rate"] = long.rate.round(2)
    viol = pd.DataFrame(viol)
    viol["rate"] = viol.rate.round(2)
    return long, viol, errors, saits_errors, coverage, run_time


def analyze(root: Path) -> dict:
    receipt = json.loads((root / "data_receipt.json").read_text())
    tickers = [a["ticker"] for a in receipt["assets"] if a["eligible"]]
    long, viol, errors, saits_errors, coverage, run_time = load(root, tickers)
    cond = condition_level(long)
    if errors:   # a failed condition is excluded for every method
        bad = pd.DataFrame(errors)[KEY + ["mask_type"]].assign(_bad=1)
        bad["rate"] = bad.rate.round(2)
        cond = cond.merge(bad, on=KEY + ["mask_type"], how="left")
        cond = cond[cond._bad.isna()].drop(columns="_bad")
    cpu_log = _read_json(root / "fold_D_cpu" / "cpu_run_log.json") or {}
    result = {"code_id": C.CODE_ID, "primary_method": PRIMARY, "fixed_setting": list(C.CFRI_FIXED),
              "eligible_assets": receipt["eligible_count"], "excluded_assets": receipt["excluded"],
              "provider_failures_accepted": receipt.get("provider_failures_accepted", False),
              "cpu_environment": cpu_log.get("environment"), "cpu_code_sha256": cpu_log.get("code_sha256"),
              "saits_assets_completed": {f"{v}_{m}": coverage.get(f"{v}_{m}", 0)
                                         for v in ("V2", "V1") for m in C.MASK_TYPES},
              "run_time": run_time,
              "saits_run_metadata": {p.name: _read_json(p) for p in sorted((root / "fold_D_saits").glob("run_metadata_*.json"))}
              if (root / "fold_D_saits").is_dir() else {},
              "cpu_condition_errors": errors, "saits_condition_errors": saits_errors, "by_mask": {}}
    for mt in sorted(cond.mask_type.unique()):
        c, l = cond[cond.mask_type == mt], long[long.mask_type == mt]
        v = viol[(viol.mask_type == mt)]
        summary, asset = summarize(c[c.method.isin(CPU_ORDER)], l[l.method.isin(CPU_ORDER)], v, primary_tests=(mt == "feature"))
        summary["saits"] = saits_comparisons(c)
        if summary["saits"]:
            sa = asset_table(c[c.method.isin(SAITS_METHODS)])
            summary["saits_mean"] = sa.mean().round(5).to_dict()
            summary["saits_assets"] = sa.notna().sum().to_dict()
            asset = asset.join(sa)    # SAITS columns: means over the conditions each SAITS run completed
        result["by_mask"][mt] = summary
        asset.to_csv(root / f"fold_D_asset_level_{mt}.csv")
    if "feature" in result["by_mask"]:
        fs = result["by_mask"]["feature"]
        primary = dict(fs["hypotheses"])
        h, comp = SAITS_HYPOTHESIS
        primary[h] = fs["saits"].get(comp) or {"comparator": comp, "evaluable": False, "criterion_met": False}
        result["primary"] = primary
        result["projection_equalized"] = {c: fs["paired_vs_cfri"].get(c) or fs["saits"].get(c)
                                          for c in PROJECTION_EQUALIZED}
    (root / "fold_D_results.json").write_text(json.dumps(result, indent=1, default=float))
    (root / "fold_D_report.md").write_text(report(result))
    return result


def report(result: dict) -> str:
    excluded = [e["ticker"] for e in result["excluded_assets"]]
    lines = [f"# CFRI confirmatory fold D results ({result['code_id']})", "",
             f"Eligible assets: {result['eligible_assets']}; excluded: {excluded or 'none'} (reasons in "
             f"data_receipt.json); CPU condition errors: {len(result['cpu_condition_errors'])}.",
             "SAITS assets completed (of " + str(result["eligible_assets"]) + "): " +
             ", ".join(f"{k} {v}" for k, v in result["saits_assets_completed"].items()) +
             f"; SAITS condition errors: {len(result['saits_condition_errors'])}.", ""]

    def row(label, r):
        if r is None or r.get("evaluable") is False or not r.get("assets"):
            return f"| {label} | {(r or {}).get('comparator', '')} | not evaluable | | | | | no |"
        return (f"| {label} | {r['comparator']} | {r['mean_method']:.4f} | {r['mean_comparator']:.4f} | "
                f"{r['mean_difference']:+.4f} ({r['boot95_low']:+.4f}, {r['boot95_high']:+.4f}) | "
                f"{r['method_lower_in_assets']}/{r['assets']} | {r['threshold_k']} | {'yes' if r['criterion_met'] else 'no'} |")

    head = ["| | comparator | CFRI mean | comparator mean | difference (95% bootstrap) | CFRI lower in | k | met |",
            "|---|---|---|---|---|---|---|---|"]
    if "primary" in result:
        lines += ["## Primary hypotheses (feature-wise masks)", ""] + head
        lines += [row(h, r) for h, r in result["primary"].items()] + [""]
        lines += ["## Projection-equalized comparisons (secondary, feature-wise masks)", ""] + head
        lines += [row(f"P{i}", r or {"comparator": c, "evaluable": False})
                  for i, (c, r) in enumerate(result["projection_equalized"].items(), 1)] + [""]
    for mt, s in result["by_mask"].items():
        lines += [f"## Mean asset-level NRMSE, {mt} masks ({s['assets']} assets, {s['conditions']} conditions per method)", "",
                  "| method | mean | median | CFRI lower in |", "|---|---|---|---|"]
        for m in s["methods"]:
            r = s["paired_vs_cfri"].get(m)
            w = f"{r['method_lower_in_assets']}/{r['assets']}" if r else ""
            lines.append(f"| {m} | {s['mean'][m]:.4f} | {s['median'][m]:.4f} | {w} |")
        for m, r in s.get("saits", {}).items():
            lines.append(f"| {m} (paired on {r['conditions_paired']} of {r['conditions_cfri']} conditions) | "
                         f"{r['mean_comparator']:.4f} | {r['median_comparator']:.4f} | {r['method_lower_in_assets']}/{r['assets']} |")
        if "ohlc_violation_share" in s:
            v = s["ohlc_violation_share"]
            lines += ["", "OHLC violation share of rows with an imputed price: " +
                      ", ".join(f"{k} {v[k]:.3f}" for k in sorted(v))]
        lines.append("")
    return "\n".join(lines)


def export(root: Path, code_dir: Path, name: str = "CFRI_fold_D_results.zip") -> Path:
    """Metrics, receipts, analysis and code only; files under raw_do_not_publish are never included."""
    zpath = root / name
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for sub in ("fold_D_cpu", "fold_D_saits"):
            if (root / sub).is_dir():
                for f in sorted((root / sub).glob("*")):
                    if f.is_file() and not f.name.endswith(".tmp"):
                        z.write(f, f"{sub}/{f.name}")
        for f in ["data_receipt.json", "fold_D_results.json", "fold_D_report.md", "code_sha256.json"] + \
                 [p.name for p in sorted(root.glob("fold_D_asset_level_*.csv"))]:
            if (root / f).exists():
                z.write(root / f, f)
        for f in sorted(code_dir.glob("*.py")):
            z.write(f, f"code/{f.name}")
        bad = [n for n in z.namelist() if "raw_do_not_publish" in n]
        if bad:
            raise RuntimeError(f"raw data in export: {bad}")
    return zpath


def code_hashes(code_dir: Path) -> dict:
    return {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted(code_dir.glob("*.py"))}
'''

FILES['run_confirm.py'] = r'''#!/usr/bin/env python3
"""Command-line driver of the CFRI confirmatory study (fold D). Steps, in order:

  python run_confirm.py --root DIR data                 acquire + QC, writes data_receipt.json (score-blind)
  python run_confirm.py --root DIR cpu [--workers N]    CPU methods, feature-wise and whole-row masks
  python run_confirm.py --root DIR saits --variant V2 --mask-type feature [--smoke 1]
  python run_confirm.py --root DIR analyze              prespecified analysis + results ZIP (no raw prices)

Every step records the SHA-256 of the code files it ran with (code_sha256.json); later steps refuse to run if
the code changed after an earlier step had started.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import os
import sys
from pathlib import Path

CODE_DIR = Path(__file__).resolve().parent
PROTOCOL_ID = "CFRI-confirmatory-protocol-v1"


def code_hashes() -> dict:
    return {f.name: hashlib.sha256(f.read_bytes()).hexdigest() for f in sorted(CODE_DIR.glob("*.py"))}


def check_code(root: Path) -> None:
    path, now = root / "code_sha256.json", code_hashes()
    if path.exists():
        before = json.loads(path.read_text())
        if before["files"] != now:
            raise SystemExit("code differs from the code used by an earlier step of this run; stopping")
    else:
        path.write_text(json.dumps({"protocol": PROTOCOL_ID, "files": now}, indent=1))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, type=Path)
    sub = ap.add_subparsers(dest="step", required=True)
    dat = sub.add_parser("data")
    dat.add_argument("--accept-provider-failures", action="store_true",
                     help="finalize the receipt although some downloads failed repeatedly (score-blind)")
    cpu = sub.add_parser("cpu")
    cpu.add_argument("--workers", type=int, default=0)
    cpu.add_argument("--mask-types", default="feature,row")
    sa = sub.add_parser("saits")
    sa.add_argument("--variant", required=True, choices=["V1", "V2"])
    sa.add_argument("--mask-type", required=True, choices=["feature", "row"])
    sa.add_argument("--smoke", type=int, default=0)
    sa.add_argument("--threads", type=int, default=2)
    sub.add_parser("analyze")
    a = ap.parse_args()
    if a.step in ("cpu", "saits"):   # bounded BLAS threads per process (several processes may run at once)
        n = "1" if a.step == "cpu" else str(a.threads)
        for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
            os.environ[var] = n
    sys.path.insert(0, str(CODE_DIR))
    root = a.root
    root.mkdir(parents=True, exist_ok=True)
    check_code(root)
    import confirm_data as D
    if a.step == "data":
        receipt = D.acquire(root)
        print(f"eligible assets: {receipt['eligible_count']} of 34; excluded: "
              f"{[e['ticker'] for e in receipt['excluded']] or 'none'} (reasons in data_receipt.json)")
        failed = D.provider_failures(receipt)
        if failed and not receipt.get("provider_failures_accepted"):
            if a.accept_provider_failures:
                D.accept_provider_failures(root)
                print("provider failures accepted as exclusions:", failed)
            else:
                print(f"Download failed for {failed} after three attempts. No method has run. Run the data step "
                      "again later; if the failure persists, rerun it with --accept-provider-failures.")
                raise SystemExit(2)
        return
    if not (root / D.RECEIPT).exists():
        raise SystemExit("run the data step first")
    receipt = json.loads((root / D.RECEIPT).read_text())
    if D.provider_failures(receipt) and not receipt.get("provider_failures_accepted"):
        raise SystemExit("some downloads failed; rerun the data step first (see data_receipt.json)")
    if a.step == "cpu":
        import confirm_cpu
        frames = D.load_frames(root)
        log = confirm_cpu.run_cpu(root, frames, tuple(a.mask_types.split(",")), a.workers or None)
        if not log["complete"]:
            raise SystemExit("CPU step incomplete; run it again to resume")
        print("CPU step complete:", log["shards"], "shards")
    elif a.step == "saits":
        import confirm_saits
        frames = D.load_frames(root)
        if a.smoke:
            confirm_saits.run_saits(root, dict(list(frames.items())[:1]), a.variant, a.mask_type,
                                    smoke_conditions=a.smoke, threads=a.threads)
        else:
            confirm_saits.run_saits(root, frames, a.variant, a.mask_type, threads=a.threads)
            print(f"SAITS {a.variant} {a.mask_type} complete")
    elif a.step == "analyze":
        import confirm_analyze
        confirm_analyze.analyze(root)    # needs only the receipt and the metric shards, not the raw data
        zpath = confirm_analyze.export(root, CODE_DIR)
        print((root / "fold_D_report.md").read_text())
        print("results ZIP:", zpath)


if __name__ == "__main__":
    main()
'''

import hashlib, pathlib
EXPECTED = {
 "cfri_core.py": "ef33fa6f0fdcb8a79b0f364f9ff00b7820a707e8323f6f03bac554de5d657721",
 "confirm_data.py": "3cedfb265cf71209ed4234d0e05e22b14a99495311ccebe06a46796413bc92db",
 "confirm_cpu.py": "fecd0fbe7a3f250b6f28ba5cea08f4c326f5799296caf0e547d52c8411179cf6",
 "confirm_saits.py": "86b592b9d380cc0fd694d35f8031f8dac16a130a5bfed022c56b180ca27f07a4",
 "confirm_analyze.py": "cccc4bb504b518b222cab4ec053f0e94dd4806652f16462c04419e320fd84eb6",
 "run_confirm.py": "99236ec45445cb6954a4ab657a2c445c6751c063f15fbdc6165d3274b9a96661"
}
CODE_DIR = pathlib.Path('/content/cfri_code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, text in FILES.items():
    data = text.encode('utf-8')
    if hashlib.sha256(data).hexdigest() != EXPECTED[name]:
        raise SystemExit(f'{name}: SHA-256 does not match the frozen protocol; stopping')
    (CODE_DIR / name).write_bytes(data)
print('Frozen code verified:', ', '.join(f'{n} {h[:12]}' for n, h in EXPECTED.items()))

## 3. Google Drive

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/CFRI_confirmatory')
ROOT.mkdir(parents=True, exist_ok=True)
LOGS = ROOT / 'logs'
LOGS.mkdir(exist_ok=True)

def step(*args):
    cmd = [sys.executable, str(CODE_DIR / 'run_confirm.py'), '--root', str(ROOT), *args]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env=dict(os.environ, PYTHONUNBUFFERED='1'))
    try:
        for line in p.stdout:
            print(line, end='')
    except KeyboardInterrupt:      # stopping the cell also stops the child process
        p.terminate()
        raise
    if p.wait() != 0:
        raise SystemExit(f'step {args} stopped (exit {p.returncode}); see the message above')

def count(pattern, sub):
    d = ROOT / sub
    return len(list(d.glob(pattern))) if d.is_dir() else 0

print('Drive folder:', ROOT)
print('Finished CPU shards:', count('*.jsonl.done.json', 'fold_D_cpu'),
      '| finished SAITS shards:', count('*.done.json', 'fold_D_saits'))

## 4. Veri ve kalite kontrolü (yöntem çalıştırılmaz)

Bazı ETF'ler için indirme üç denemede de başarısız olursa notebook burada durur ve hiçbir yöntem çalışmaz.
Birkaç dakika (gerekirse birkaç saat) sonra yeniden **Run all** yapın; yalnızca başarısız olanlar tekrar indirilir.
Hata kalıcıysa Claude'a haber verin. Yalnızca birlikte karar verirsek alttaki `ACCEPT_PROVIDER_FAILURES` değeri
`True` yapılır ve o ETF'ler protokoldeki kurala göre dışlanır.

In [ ]:
ACCEPT_PROVIDER_FAILURES = False   # True only after deciding it together (persistent download failures)
step('data', *(['--accept-provider-failures'] if ACCEPT_PROVIDER_FAILURES else []))

## 5. CPU yöntemleri

In [ ]:
step('cpu', '--workers', str(os.cpu_count()))

## 6. SAITS (GPU)

Önce tek koşulluk duman testi, sonra dört iş (V2 ve V1, iki maske türü). İşlemci sayısına göre en fazla dört süreç
aynı anda çalışır. Her dakika ilerleme yazdırılır; günlükler `CFRI_confirmatory/logs` klasöründedir.

In [ ]:
gpu = subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")'],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit('GPU yok: Runtime > Change runtime type > GPU seçin ve Run all yapın')
print('GPU:', gpu, '| CPU:', os.cpu_count())
step('saits', '--variant', 'V1', '--mask-type', 'feature', '--smoke', '1')

In [ ]:
JOBS = [('V2', 'feature'), ('V1', 'feature'), ('V2', 'row'), ('V1', 'row')]
n_assets = json.loads((ROOT / 'data_receipt.json').read_text())['eligible_count']
PARALLEL = max(1, min(4, (os.cpu_count() or 2) // 2))
done = lambda v, m: count(f'*_{m}_saits_{v}.done.json', 'fold_D_saits')
queue, running, t0 = list(JOBS), {}, time.time()   # finished shards are verified and skipped by each job
print(f'{len(queue)} SAITS jobs, up to {PARALLEL} at a time')
try:
    while queue or running:
        while queue and len(running) < PARALLEL:
            v, m = queue.pop(0)
            log = open(LOGS / f'saits_{v}_{m}.log', 'a')
            cmd = [sys.executable, str(CODE_DIR / 'run_confirm.py'), '--root', str(ROOT), 'saits',
                   '--variant', v, '--mask-type', m, '--threads', '2']
            running[(v, m)] = (subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                                env=dict(os.environ, PYTHONUNBUFFERED='1')), log)
        time.sleep(60)
        for (v, m), (p, log) in list(running.items()):
            if p.poll() is not None:
                log.close()
                del running[(v, m)]
                if p.returncode != 0:
                    print(open(LOGS / f'saits_{v}_{m}.log').read()[-3000:])
                    raise SystemExit(f'SAITS {v} {m} stopped (exit {p.returncode}); Run all again to resume')
        print(f'{(time.time() - t0) / 60:6.1f} min | ' + ' | '.join(f'{v}-{m}: {done(v, m)}/{n_assets}' for v, m in JOBS),
              flush=True)
finally:
    for p, log in running.values():   # on an error or interruption, stop the other SAITS processes
        if p.poll() is None:
            p.terminate()
print('All SAITS runs complete.')

## 7. Önceden belirlenmiş analiz ve sonuç ZIP'i

In [ ]:
step('analyze')
import shutil
from google.colab import files
zip_path = Path('/content/CFRI_fold_D_results.zip')
shutil.copy(ROOT / 'CFRI_fold_D_results.zip', zip_path)
files.download(str(zip_path))